# Video masks

COLMAP feature-extraction masks for every roll video, of both kinds:

- **`edited_vid`** — has a burned-in stats overlay (Garmin/GoPro gauges, text, course loop) *and*
  the buggy shell in frame. Needs both masked.
- **`video_preview`** — clean recording, so only the shell needs masking.

White = keep, black = exclude, consumed as `<mask_path>/<image name>.png` (see `link_masks` in
`colab/hloc.ipynb`). Where a roll has a clean preview, its edited version is skipped.

Everything is derived from frames sampled while the buggy is moving, and reviewed on the
**temporal median** of those frames: overlay graphics and the shell stay sharp there while the
scene blurs away, and the shell's blur band shows its full drift under mount motion.
Sampling is restricted to the roll's own window (`rollfile.local_start_ms/local_end_ms`) when
the database has one — some recordings run 27 minutes with a 2-minute roll inside, and
sampling the whole file lands almost entirely on idle footage.

**Overlays (automatic).** Per-pixel median gradient vector: screen-fixed graphics keep the same
gradient every frame, scene gradients vary in sign and cancel. Hysteresis thresholds, a faint
second pass for translucent panel borders that touch their own text, corner snapping for
edge-anchored panels, and each element becomes a filled bounding box. Reviewed in
`overlays.html`, where you can box a miss or delete a false positive.

**Shell (manual).** Everything below a horizontal cut line, placed by hand in `shells.html` —
one click per video. Auto-detection was tried and removed: flow finds the shell on opaque hulls
but not on mirror-finish canopies or flat untextured ones, and a wrong proposal costs more to
check than a line costs to place.

**Outputs.** Detection cache in `data/cache/masks/`; final masks (overlay *and* shell cut) in
`data/masks/<file_id>.png` at native resolution, with `data/masks/masks.json` listing every
video (file id, roll id, file name, ignore flag, `overlay_frac`, `mask_frac`).

In [ ]:
%env DATA_PATH=../../../data
import json
import os
import re
import sqlite3
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
from tqdm.notebook import tqdm

from lib.paths import resolve_path

cv2.setNumThreads(4)
DATA = Path(os.environ['DATA_PATH'])
MASKS = DATA / 'cache' / 'masks'          # detection cache
OUT = DATA / 'masks'                      # final masks + manifest
ANNOT = MASKS / 'annotations.json'
REVIEW = Path('../../../tmp/mask_review')

ONLY = None            # set to a list of file ids to (re)process just those
N_SAMPLES = 50         # sampled positions per video
DENSE_MULT = 4         # retry this much denser when too few samples were moving
MIN_SAMPLES = 8
FLOW_DT_S = 0.3        # flow baseline: scene parallax >> shell wobble
MOVE_PX = 12.0         # p90 bottom-half flow below this = not moving, sample skipped
FAST_K = 30            # statistics use only the fastest K samples
PROC_W = 640
DEFAULT_CUT = 0.78     # where the shell line starts in review, before you move it

EDGE_HI, EDGE_LO = 0.6, 0.35
EDGE_FAINT = 0.20      # translucent panel borders are far fainter than their text; accept them
                       # only where they touch something already detected
MIN_AREA = 35          # smallest overlay component kept (elevation text is ~40 px)
JOIN = 25              # close overlay parts this far apart before boxing them
PAD = 6                # grow each overlay box by this many px
BORDER = 10            # look this far in from each edge for encoder border lines
SNAP = 0.12            # a box this close to a frame edge is a corner-anchored panel
MAX_COMP_FRAC = 0.06   # weak persistence components above this fraction = scene band

# 2015-2016 speed dial: its arc sweeps outside the ring and cannot be detected from a median,
# so the bottom-left box is grown to this corner (measured by hand on file #124)
DIAL_X, DIAL_Y = 0.173, 0.674

# not filmed from the buggy: "FC" = follow car; the ids are what review turned up
EXCLUDE_RE = re.compile(r'(^|[^a-z])(fc|rear|crotch)([^a-z]|$)', re.I)
EXCLUDE_IDS = {84, 85, 86, 87, 88, 89, 106, 154, 155, 163, 164, 234, 238, 308, 309,
               630, 876, 1004, 1047, 1092, 3146}

In [ ]:
db = sqlite3.connect(DATA / 'db' / 'srs.db')
rows = db.execute('''
    SELECT f.id, f.uri, f.type, MIN(rd.year), GROUP_CONCAT(DISTINCT r.id),
      MIN(rf.local_start_ms), MAX(rf.local_end_ms),
      MIN(CASE WHEN EXISTS (SELECT 1 FROM rollfile rf2 JOIN file f2 ON f2.id = rf2.file_id
                            WHERE rf2.roll_id = r.id AND f2.type LIKE 'video_preview%')
               THEN 1 ELSE 0 END)
    FROM file f JOIN rollfile rf ON rf.file_id = f.id
    JOIN roll r ON r.id = rf.roll_id JOIN rolldate rd ON rd.id = r.roll_date_id
    WHERE f.type LIKE 'edited_vid%' OR f.type LIKE 'video_preview%'
    GROUP BY f.id''').fetchall()
db.close()

videos, missing, has_preview = [], 0, 0
for fid, uri, ftype, year, rolls, t0, t1, all_have_preview in rows:
    p = Path(resolve_path(uri))
    kind = 'preview' if ftype.startswith('video_preview') else 'edited'
    if not p.exists():
        missing += 1
        continue
    if kind == 'edited':
        if fid in EXCLUDE_IDS or EXCLUDE_RE.search(p.stem):
            continue
        if all_have_preview:      # the roll has a clean preview; mask that instead
            has_preview += 1
            continue
    videos.append({'id': fid, 'uri': uri, 'path': p, 'year': year, 'kind': kind,
                   'rolls': rolls, 'window': (t0, t1) if t0 is not None and t1 else None})
videos.sort(key=lambda v: (v['kind'], v['year'], v['id']))
if ONLY:
    videos = [v for v in videos if v['id'] in set(ONLY)]
print(f'{len(videos)} videos to mask ({missing} not on disk, {has_preview} edited skipped: '
      f'roll has a clean preview)')
print('  by kind:', dict(Counter(v['kind'] for v in videos)),
      '| with a roll window:', sum(1 for v in videos if v['window']))

In [ ]:
def k(size):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size | 1, size | 1))


def sample_points(n, fps, window, mult=1):
    """Frame indices to sample: the roll's own window when known, else the middle 80%."""
    count = N_SAMPLES * mult
    if window:
        f0, f1 = int(window[0] / 1000 * fps), int(window[1] / 1000 * fps)
        f0, f1 = max(0, f0), min(n, f1)
        if f1 - f0 > 2 * count:
            return np.linspace(f0, f1, count)
    return np.linspace(0.1, 0.9, count) * n


def video_stats(path, window, want_persist):
    """Median frame + mid frame (+ gradient persistence for overlay detection)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    if n < 60:
        return None
    dt = max(1, round(FLOW_DT_S * fps))
    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_FAST)

    for mult in (1, DENSE_MULT):          # retry denser if the roll window was wrong/missing
        samples, colors = [], []
        for t in sample_points(n, fps, window, mult):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(t))
            ok, a = cap.read()
            ok2, b = False, None
            for _ in range(dt):
                ok2, b = cap.read()
            if not (ok and ok2):
                continue
            h, w = a.shape[:2]
            sc = PROC_W / w
            small = cv2.resize(a, None, fx=sc, fy=sc)
            ga = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            gb = cv2.resize(cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), None, fx=sc, fy=sc)
            flow = dis.calc(ga, gb, None)
            mag = np.hypot(flow[..., 0], flow[..., 1])
            speed = np.percentile(mag[mag.shape[0] // 2:], 90)
            if speed < MOVE_PX:
                continue
            colors.append(small)
            f = ga.astype(np.float32) / 255
            samples.append((speed, cv2.Sobel(f, cv2.CV_32F, 1, 0),
                            cv2.Sobel(f, cv2.CV_32F, 0, 1)) if want_persist else (speed, None, None))
        if len(samples) >= MIN_SAMPLES:
            break
    cap.release()
    if len(samples) < MIN_SAMPLES:
        return None
    order = np.argsort([-s[0] for s in samples])[:FAST_K]
    out = {'med': np.median([colors[i] for i in order], 0).astype(np.uint8),
           'mid': colors[len(colors) // 2], 'native': (w, h), 'n_samples': len(order)}
    if want_persist:
        out['persist'] = np.hypot(np.median([samples[i][1] for i in order], 0),
                                  np.median([samples[i][2] for i in order], 0))
    return out


def strip_border_lines(m):
    """Drop the 2-3 px static line some encoders leave along a frame edge: it is genuinely
    static, so it detects, and it bridges unrelated widgets into one full-height component."""
    h, w = m.shape
    for c in list(range(BORDER)) + list(range(w - BORDER, w)):
        if m[:, c].mean() > 0.4:
            m[:, c] = False
    for r in list(range(BORDER)) + list(range(h - BORDER, h)):
        if m[r].mean() > 0.4:
            m[r] = False
    return m


def overlay_mask(persist):
    """Detected overlay edges -> filled bounding boxes (elements are rectangular UI)."""
    strong, weak = persist > EDGE_HI, persist > EDGE_LO
    nl, lbl, st, _ = cv2.connectedComponentsWithStats(weak.astype(np.uint8))
    m = np.zeros_like(strong)
    for i in range(1, nl):
        sel = lbl == i
        if not strong[sel].any():
            continue
        m |= sel if st[i, 4] < MAX_COMP_FRAC * lbl.size else (sel & strong)
    h, w = m.shape
    # faint panel outlines: keep small ones that touch an accepted element
    nl, lbl, st, _ = cv2.connectedComponentsWithStats((persist > EDGE_FAINT).astype(np.uint8))
    near = cv2.dilate(m.astype(np.uint8), k(5)) > 0
    for i in range(1, nl):
        sel = lbl == i
        if st[i, 4] <= 0.03 * m.size and near[sel].any():
            m |= sel
    m = strip_border_lines(m)
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE, k(JOIN))
    nl, lbl, st, _ = cv2.connectedComponentsWithStats(m)
    boxes = [[x, y, x + cw, y + ch] for x, y, cw, ch, a in st[1:] if a >= MIN_AREA]
    for _ in range(3):            # merge boxes that touch or overlap
        merged, used = [], [False] * len(boxes)
        for i, b in enumerate(boxes):
            if used[i]:
                continue
            for j in range(i + 1, len(boxes)):
                c = boxes[j]
                if used[j] or b[0] > c[2] + PAD or c[0] > b[2] + PAD \
                        or b[1] > c[3] + PAD or c[1] > b[3] + PAD:
                    continue
                b = [min(b[0], c[0]), min(b[1], c[1]), max(b[2], c[2]), max(b[3], c[3])]
                used[j] = True
            merged.append(b)
        if len(merged) == len(boxes):
            break
        boxes = merged
    out = np.zeros((h, w), np.uint8)
    for x0, y0, x1, y1 in boxes:
        x0, x1 = (0 if x0 < SNAP * w else x0), (w if w - x1 < SNAP * w else x1)
        y0, y1 = (0 if y0 < SNAP * h else y0), (h if h - y1 < SNAP * h else y1)
        out[max(0, y0 - PAD):y1 + PAD, max(0, x0 - PAD):x1 + PAD] = 1
    return out


def expand_dial(ov, year):
    """2015-2016 speed dial: grow the bottom-left box to the corner measured on #124."""
    if year not in (2015, 2016):
        return ov
    h, w = ov.shape
    nl, lbl, st, _ = cv2.connectedComponentsWithStats(ov.astype(np.uint8))
    for x, y, cw, ch, _ in st[1:]:
        if (y + ch / 2) / h > 0.6 and (x + cw / 2) / w < 0.3:
            ov[min(y, int(DIAL_Y * h)):y + ch, x:max(x + cw, int(DIAL_X * w))] = 1
            break
    return ov


def apply_edits(ov, edits):
    """Review edits, in order: 'a' add box, 'd' clear box, 'dp' drop the clicked shape."""
    h, w = ov.shape
    for e in edits or []:
        b = e['b']
        if e['t'] == 'dp':
            lbl = cv2.connectedComponents(ov.astype(np.uint8))[1]
            v = lbl[min(int(b[1] * h), h - 1), min(int(b[0] * w), w - 1)]
            if v:
                ov[lbl == v] = 0
        else:
            ov[int(b[1] * h):int(b[3] * h), int(b[0] * w):int(b[2] * w)] = e['t'] == 'a'
    return ov


def ov_img(med, ov):
    out = med.copy()
    tint = out.copy()
    tint[ov > 0] = (0, 0, 255)
    out = cv2.addWeighted(tint, 0.4, out, 0.6, 0)
    cs, _ = cv2.findContours(ov.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, cs, -1, (0, 0, 255), 2)
    return out

In [ ]:
for sub in ('med', 'mid', 'ov'):
    (REVIEW / sub).mkdir(parents=True, exist_ok=True)
MASKS.mkdir(parents=True, exist_ok=True)
for v in tqdm(videos):
    meta_p = MASKS / f'{v["id"]}.json'
    if meta_p.exists() and not ONLY:
        continue
    edited = v['kind'] == 'edited'
    stats = video_stats(v['path'], v['window'], want_persist=edited)
    meta = {'uri': v['uri'], 'year': v['year'], 'name': v['path'].name, 'kind': v['kind'],
            'rolls': v['rolls'], 'windowed': bool(v['window'])}
    if stats is None:
        meta['flag'] = 'unreadable_or_static'
    else:
        cv2.imwrite(str(REVIEW / 'med' / f'{v["id"]}.jpg'), stats['med'])
        cv2.imwrite(str(REVIEW / 'mid' / f'{v["id"]}.jpg'), stats['mid'])
        meta |= {'n_samples': stats['n_samples'], 'native': list(stats['native'])}
        if edited:
            ov = overlay_mask(stats['persist'])
            cv2.imwrite(str(MASKS / f'{v["id"]}.persist.png'),
                        np.clip(stats['persist'] * 100, 0, 255).astype(np.uint8))
            cv2.imwrite(str(MASKS / f'{v["id"]}.ov.png'), ov * 255)
            cv2.imwrite(str(REVIEW / 'ov' / f'{v["id"]}.jpg'), ov_img(stats['med'], ov))
            meta['overlay_frac'] = round(float(ov.mean()), 4)
    meta_p.write_text(json.dumps(meta))

In [ ]:
# Re-derive overlay masks from cached persistence — run after changing EDGE_*/JOIN/PAD/SNAP.
n = 0
for p in sorted(MASKS.glob('*.persist.png')):
    fid = p.name.split('.')[0]
    persist = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 100
    ov = overlay_mask(persist)
    cv2.imwrite(str(MASKS / f'{fid}.ov.png'), ov * 255)
    med = cv2.imread(str(REVIEW / 'med' / f'{fid}.jpg'))
    if med is not None:
        cv2.imwrite(str(REVIEW / 'ov' / f'{fid}.jpg'), ov_img(med, ov))
    meta_p = MASKS / f'{fid}.json'
    meta = json.loads(meta_p.read_text())
    meta['overlay_frac'] = round(float(ov.mean()), 4)
    meta_p.write_text(json.dumps(meta))
    n += 1
print(f'{n} overlay masks re-derived from cache')

In [ ]:
COMMON = r"""
body { font: 13px/1.4 sans-serif; background: #111; color: #ddd; margin: 0; padding: 10px }
#top { display: flex; gap: 14px; align-items: center; flex-wrap: wrap; margin-bottom: 6px }
#wrap { position: relative; width: min(1100px, 96vw) }
#pair { display: flex; gap: 8px; width: min(1500px, 98vw) }
.pane { position: relative; flex: 1 }
.pane img { width: 100%; display: block; border-radius: 4px }
.pane .line { position: absolute; left: 0; right: 0; border-top: 2px solid #f4d03f }
.pane .line.set { border-color: #2ecc71 }
.pane .shade { position: absolute; left: 0; right: 0; bottom: 0;
               background: rgba(40,120,255,.28) }
.pane .tag { position: absolute; left: 4px; bottom: 2px; font-size: 11px; color: #fff;
             text-shadow: 0 0 4px #000 }
#wrap img { width: 100%; display: block; border-radius: 4px }
#line { position: absolute; left: 0; right: 0; border-top: 2px dashed #f4d03f;
        pointer-events: none; z-index: 3 }
#line.set { border-color: #2ecc71 }
#shade { position: absolute; left: 0; right: 0; bottom: 0; background: rgba(40,120,255,.25);
         pointer-events: none }
#cv { position: absolute; inset: 0; cursor: crosshair; width: 100%; height: 100% }
#thumbs { display: flex; gap: 6px; margin-top: 6px; flex-wrap: wrap }
.th { position: relative; width: 170px }
.th img { width: 100%; border-radius: 3px; display: block }
.th div { position: absolute; left: 0; right: 0; border-top: 2px solid #f4d03f }
.th span { position: absolute; left: 2px; top: 0; color: #fff; font-size: 11px;
           text-shadow: 0 0 3px #000 }
b.ok { color: #6c6 } b.no { color: #f66 } b.todo { color: #888 }
kbd { background: #333; border-radius: 3px; padding: 1px 5px }
#help { color: #999; margin-top: 8px; line-height: 1.9 }
button { background: #333; color: #ddd; border: 1px solid #555; border-radius: 3px;
         padding: 3px 8px; cursor: pointer }
a { color: #89f }
"""

SHARED_JS = r"""
let ann = Object.assign({}, SAVED, JSON.parse(localStorage.getItem("maskann") || "{}"));
let i = 0;
function store() { localStorage.setItem("maskann", JSON.stringify(ann)); }
function save() {
  const b = new Blob([JSON.stringify(ann, null, 1)], {type: "application/json"});
  const a = document.createElement("a");
  a.href = URL.createObjectURL(b); a.download = "annotations.json"; a.click();
}
function next(d) { i = Math.min(CARDS.length - 1, Math.max(0, i + d)); onmove(); show(); }
"""

SHELL_HTML = r"""<!doctype html><meta charset="utf-8"><title>shell line</title>
<style>__COMMON__</style>
<div id="top">
  <div id="pos"></div><div id="meta"></div><div id="state"></div>
  <button onclick="save()">export annotations.json</button>
  <a href="overlays.html">overlay review &rarr;</a>
</div>
<div id="pair">
  <div class="pane"><img id="imA"><div class="shade"></div><div class="line"></div>
    <span class="tag">median of the roll (shell sharp, scene blurred)</span></div>
  <div class="pane"><img id="imB"><div class="shade"></div><div class="line"></div>
    <span class="tag">single frame, mid-roll</span></div>
</div>
<div id="help">
Click either image to put the line above the shell &mdash; blue is what gets masked. The median
(left) shows the shell's full drift as a blur band; the frame (right) shows it sharp.<br>
<kbd>click</kbd> set line &nbsp; <kbd>&uarr;</kbd><kbd>&darr;</kbd> nudge
(<kbd>shift</kbd> bigger) &nbsp; <kbd>a</kbd>/<kbd>Enter</kbd> accept &amp; next &nbsp;
<kbd>n</kbd> no shell visible &nbsp; <kbd>x</kbd> ignore this video &nbsp;
<kbd>&larr;</kbd><kbd>&rarr;</kbd> prev/next &nbsp; <kbd>?</kbd> next unreviewed
</div>
<script>
const CARDS = __CARDS__, SAVED = __SAVED__;
__SHARED__
let cut = null;
function cur() { return CARDS[i]; }
function rec(id) { return ann[String(id)] || {}; }
function onmove() { cut = null; }
function show() {
  const c = cur(), a = rec(c.id);
  if (cut === null) {
    cut = a.cut !== undefined ? a.cut
        : (i > 0 && rec(CARDS[i - 1].id).cut !== undefined) ? rec(CARDS[i - 1].id).cut
        : c.cut;
  }
  document.getElementById("imA").src = "med/" + c.id + ".jpg";
  document.getElementById("imB").src = "mid/" + c.id + ".jpg";
  document.getElementById("pos").textContent = (i + 1) + "/" + CARDS.length;
  document.getElementById("meta").innerHTML = "<b>" + c.year + "</b> roll <b>" +
    (c.rolls || "?") + "</b> &middot; file #" + c.id + " " + c.name +
    (c.kind === "preview" ? " <b style=color:#89f>preview</b>" : "") + " " +
    c.flags.map(f => "<b class=no>" + f + "</b>").join(" ");
  document.getElementById("state").innerHTML =
    a.verdict === "exclude" ? "<b class=no>ignored</b>"
    : a.verdict === "no_nose" ? "<b class=ok>no shell</b>"
    : a.cut !== undefined ? "<b class=ok>set " + Math.round(a.cut * 100) + "%</b>"
    : "<b class=todo>" + Math.round(cut * 100) + "%</b>";
  const hide = a.verdict === "exclude" || cut >= 1;
  for (const el of document.querySelectorAll(".line, .shade")) {
    el.style.display = hide ? "none" : "block";
    el.style.top = (cut * 100) + "%";
  }
  for (const el of document.querySelectorAll(".line"))
    el.className = "line" + (a.cut !== undefined ? " set" : "");
}
document.getElementById("pair").onclick = e => {
  const img = e.target.closest(".pane").querySelector("img");
  const r = img.getBoundingClientRect();
  cut = Math.min(1, Math.max(0, (e.clientY - r.top) / r.height));
  show();
};
function set(v) {
  const c = cur();
  ann[String(c.id)] = Object.assign(rec(c.id), v);
  for (const key of ["cut", "verdict"]) if (!(key in v)) delete ann[String(c.id)][key];
  store(); next(1);
}
onkeydown = e => {
  if (e.key === "ArrowRight") next(1);
  else if (e.key === "ArrowLeft") next(-1);
  else if (e.key === "ArrowUp" || e.key === "ArrowDown") {
    e.preventDefault();
    cut = Math.min(1, Math.max(0, cut + (e.key === "ArrowUp" ? -1 : 1)
                                       * (e.shiftKey ? 0.02 : 0.005)));
    show();
  } else if (e.key === "a" || e.key === "Enter") set({cut: Math.round(cut * 1000) / 1000});
  else if (e.key === "n") set({verdict: "no_nose"});
  else if (e.key === "x") set({verdict: "exclude"});
  else if (e.key === "?") {
    const j = CARDS.findIndex(c => rec(c.id).cut === undefined && !rec(c.id).verdict);
    if (j >= 0) { i = j; onmove(); show(); }
  }
};
show();
</script>"""

OV_HTML = r"""<!doctype html><meta charset="utf-8"><title>overlay review</title>
<style>__COMMON__</style>
<div id="top">
  <div id="pos"></div><div id="meta"></div><div id="state"></div>
  <button onclick="save()">export annotations.json</button>
  <a href="shells.html">shell lines &rarr;</a>
</div>
<div id="wrap"><img id="im"><canvas id="cv"></canvas><div id="line"></div></div>
<div id="help">
Red = detected overlay on the median frame; anything sharp and un-boxed is a miss.
<b>Below the yellow line the shell mask already covers everything &mdash; ignore detections
there.</b><br>
<kbd>a</kbd> looks right &nbsp; <kbd>f</kbd> flag as wrong &nbsp;
<kbd>d</kbd> toggle add/delete (or hold <kbd>shift</kbd> to delete) &nbsp;
<kbd>u</kbd> undo edit &nbsp; <kbd>&larr;</kbd><kbd>&rarr;</kbd> prev/next &nbsp;
<kbd>?</kbd> next unreviewed<br>
<b style="color:#6c6">add</b>: drag a box over a missed element.
<b style="color:#f66">delete</b>: drag a box to clear it, or <i>click</i> a bad blob to remove
that whole detected shape.
</div>
<script>
const CARDS = __CARDS__, SAVED = __SAVED__;
__SHARED__
const im = document.getElementById("im"), cv = document.getElementById("cv");
const ctx = cv.getContext("2d");
let box = null, delMode = false, shifted = false;
function cur() { return CARDS[i]; }
function onmove() { box = null; }
function rec() { return ann[String(cur().id)] || (ann[String(cur().id)] = {}); }
function edits() { const r = rec(); return (r.ov_edits = r.ov_edits || []); }
function fit() { cv.width = im.clientWidth; cv.height = im.clientHeight; }
function show() {
  const c = cur();
  im.src = "ov/" + c.id + ".jpg";
  const cutv = (ann[String(c.id)] || {}).cut ?? c.cut;
  const ln = document.getElementById("line");
  ln.style.display = cutv >= 1 ? "none" : "block";
  ln.style.top = (cutv * 100) + "%";
  document.getElementById("pos").textContent = (i + 1) + "/" + CARDS.length;
  document.getElementById("meta").innerHTML = "<b>" + c.year + "</b> #" + c.id + " " + c.name +
    " &mdash; overlay " + Math.round((c.ov || 0) * 100) + "% &nbsp; mode: " +
    (delMode ? "<b class=no>delete</b>" : "<b class=ok>add</b>");
  const a = ann[String(c.id)] || {}, es = a.ov_edits || [];
  const na = es.filter(e => e.t === "a").length, nd = es.length - na;
  document.getElementById("state").innerHTML = a.ov_bad ? "<b class=no>flagged</b>"
    : es.length ? "<b class=ok>+" + na + " / -" + nd + "</b>"
    : a.ov_ok ? "<b class=ok>ok</b>" : "<b class=todo>unreviewed</b>";
  fit(); redraw();
}
function rect(b, col, dash) {
  ctx.strokeStyle = col;
  ctx.setLineDash(dash ? [6, 4] : []);
  ctx.strokeRect(b[0] * cv.width, b[1] * cv.height,
                 (b[2] - b[0]) * cv.width, (b[3] - b[1]) * cv.height);
  ctx.setLineDash([]);
}
function redraw() {
  ctx.clearRect(0, 0, cv.width, cv.height);
  ctx.lineWidth = 2;
  for (const e of ((ann[String(cur().id)] || {}).ov_edits || [])) {
    if (e.t === "a") rect(e.b, "#0f0");
    else if (e.t === "d") rect(e.b, "#f44", true);
    else {                                   // point delete: cross marker
      const x = e.b[0] * cv.width, y = e.b[1] * cv.height;
      ctx.strokeStyle = "#f44";
      ctx.beginPath();
      ctx.moveTo(x - 7, y - 7); ctx.lineTo(x + 7, y + 7);
      ctx.moveTo(x + 7, y - 7); ctx.lineTo(x - 7, y + 7);
      ctx.stroke();
    }
  }
  if (box) rect(box, (delMode || shifted) ? "#f44" : "#ff0", delMode || shifted);
}
function pos(e) {
  const r = cv.getBoundingClientRect();
  return [(e.clientX - r.left) / r.width, (e.clientY - r.top) / r.height];
}
cv.onmousedown = e => {
  const p = pos(e); shifted = e.shiftKey; box = [p[0], p[1], p[0], p[1]];
};
cv.onmousemove = e => { if (box) { const p = pos(e); box[2] = p[0]; box[3] = p[1]; redraw(); } };
cv.onmouseup = () => {
  if (!box) return;
  const b = [Math.min(box[0], box[2]), Math.min(box[1], box[3]),
             Math.max(box[0], box[2]), Math.max(box[1], box[3])];
  const del = delMode || shifted;
  box = null; shifted = false;
  if (b[2] - b[0] < 0.005 && b[3] - b[1] < 0.005) {
    if (del) { edits().push({t: "dp", b: [b[0], b[1]]}); store(); }   // click = drop that shape
  } else {
    edits().push({t: del ? "d" : "a", b: b}); store();
  }
  show();
};
onkeydown = e => {
  if (e.key === "ArrowRight") next(1);
  else if (e.key === "ArrowLeft") next(-1);
  else if (e.key === "a") { rec().ov_ok = true; store(); next(1); }
  else if (e.key === "f") { rec().ov_bad = true; store(); next(1); }
  else if (e.key === "d") { delMode = !delMode; show(); }
  else if (e.key === "u") { edits().pop(); store(); show(); }
  else if (e.key === "?") {
    const j = CARDS.findIndex(c => !ann[String(c.id)]); if (j >= 0) { i = j; onmove(); show(); }
  }
};
onresize = () => { fit(); redraw(); };
im.onload = () => { fit(); redraw(); };
show();
</script>"""

SHELL_HTML = SHELL_HTML.replace('__COMMON__', COMMON).replace('__SHARED__', SHARED_JS)
OV_HTML = OV_HTML.replace('__COMMON__', COMMON).replace('__SHARED__', SHARED_JS)

In [ ]:
metas = {}
for p in MASKS.glob('*.json'):
    if p.name == 'annotations.json':
        continue
    m = json.loads(p.read_text())
    m['id'] = int(p.stem)
    metas[m['id']] = m

order = sorted(metas.values(), key=lambda m: (m.get('kind', 'edited'), m['year'], m['id']))
shell_cards = [{'id': m['id'], 'year': m['year'], 'rolls': m.get('rolls'), 'cut': DEFAULT_CUT,
                'name': m['name'], 'kind': m.get('kind', 'edited'),
                'flags': [m['flag']] if m.get('flag') else []}
               for m in order if (REVIEW / 'med' / f'{m["id"]}.jpg').exists()]
ov_cards = [c | {'ov': metas[c['id']].get('overlay_frac')} for c in shell_cards
            if c['kind'] == 'edited' and (REVIEW / 'ov' / f'{c["id"]}.jpg').exists()]

saved = json.loads(ANNOT.read_text()) if ANNOT.exists() else {}
(REVIEW / 'shells.html').write_text(
    SHELL_HTML.replace('__CARDS__', json.dumps(shell_cards))
              .replace('__SAVED__', json.dumps(saved)))
(REVIEW / 'overlays.html').write_text(
    OV_HTML.replace('__CARDS__', json.dumps(ov_cards)).replace('__SAVED__', json.dumps(saved)))
print(f'shell lines : {REVIEW / "shells.html"} ({len(shell_cards)} videos)')
print(f'overlays    : {REVIEW / "overlays.html"} ({len(ov_cards)} edited videos)')
print('export from either page (shared store), save as:', ANNOT)

In [ ]:
annots = json.loads(ANNOT.read_text()) if ANNOT.exists() else {}
OUT.mkdir(parents=True, exist_ok=True)
manifest, written = [], 0
for fid, m in sorted(metas.items()):
    a = annots.get(str(fid), {})
    verdict = a.get('verdict')
    note, keep, ov_frac, mask_frac = verdict or 'ok', None, None, None
    if verdict in ('exclude', 'not_onboard'):
        note = 'excluded'
    elif 'native' not in m:
        note = m.get('flag', 'no_frames')
    elif 'cut' not in a and verdict != 'no_nose':
        note = 'unannotated'
    else:
        w, h = m['native']
        if m.get('kind', 'edited') == 'edited':
            ov = cv2.imread(str(MASKS / f'{fid}.ov.png'), cv2.IMREAD_GRAYSCALE) > 0
            ov = apply_edits(expand_dial(ov.astype(np.uint8), m['year']), a.get('ov_edits')) > 0
        else:
            ov = np.zeros((max(1, round(PROC_W * h / w)), PROC_W), bool)
        ov_frac = round(float(ov.mean()), 4)
        ov[int((1.0 if verdict == 'no_nose' else a['cut']) * ov.shape[0]):] = True
        keep = cv2.resize((~ov).astype(np.uint8) * 255, (w, h),
                          interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(str(OUT / f'{fid}.png'), keep)
        mask_frac = round(float((keep == 0).mean()), 4)   # overlay + shell, both masked
        (MASKS / f'{fid}.json').write_text(
            json.dumps({k: v for k, v in m.items() if k != 'id'} | {'mask_frac': mask_frac}))
        written += 1
    manifest.append({'file_id': fid,
                     'roll_id': int(m['rolls'].split(',')[0]) if m.get('rolls') else None,
                     'file_name': m['name'], 'ignore': keep is None,
                     'kind': m.get('kind', 'edited'), 'note': note,
                     'overlay_frac': ov_frac, 'mask_frac': mask_frac})
(OUT / 'masks.json').write_text(json.dumps(manifest, indent=1))
print(f'{written} masks -> {OUT}, {sum(r["ignore"] for r in manifest)} ignored, '
      f'{len(manifest)} manifest rows')
print('notes:', dict(Counter(r['note'] for r in manifest)))
print('mask_frac p10/50/90:', np.round(np.percentile(
    [r['mask_frac'] for r in manifest if r['mask_frac'] is not None], [10, 50, 90]), 3))

## Using the masks in mapping

`data/masks/<file_id>.png` is the final mask at native resolution, white = keep. In
`colab/hloc.ipynb`, extend `link_masks` to compose per run rather than always linking
`mask0.png`: `m = np.minimum(mask0, cv2.resize(video_mask, frame_size))`, and skip any file whose
manifest row has `ignore: true`.

**Feature budget**: COLMAP applies masks by dropping keypoints *after* extraction (verified — a
masked frame keeps exactly the unmasked frame's keypoints outside the mask), so a masked video
loses features rather than redistributing them. Restore parity per video at extraction:
`max_num_features = min(6000, round(4096 / (1 - masked_fraction)))` when the masked fraction
exceeds ~0.15. Extraction cost is unchanged (top-k selection); matching cost grows with the
square of the count, which is why the bump is capped.

Re-running is cheap: detection is cached per file id, so the review sheets and the mask/manifest
cell rerun in seconds. Set `ONLY = [ids]` to re-detect specific videos, or delete their
`<file_id>.json`.